# Pulse 2026: Two-Month Block Pooling with CDF Merging

This notebook builds on the CDF-merging approach:

**CDF-space pooling.** Within each block we merge the two monthly distributions using
  the sample-size-weighted average-of-CDFs method. The pooled CDF equals the weighted average of the
  individual CDFs evaluated at the same x-value, not the average of the same quantile
  level.

**Closed-form confidence intervals.** Each block's IQB point estimate is accompanied
  by a 95% CI derived from the standard error of a sample quantile.

## Data

24 months of M-Lab data (January 2024 – December 2025), pre-aggregated to country-level
percentiles (p1, p5, p10, p25, p50, p75, p90, p95, p99) for download, upload, latency,
and loss.  IQB scores are computed at each of those 9 percentile levels per country-month.

## 1. Imports and Configuration

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq
import time

from iqb import IQBCalculator

calculator = IQBCalculator()

DATA_DIR = Path("./data/v1")

# IQB is computed at all 9 percentile levels so we can reconstruct the full CDF shape.
# p25/p50/p75 are the "headline" levels used for reporting; the others anchor the tails.
PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
PROBS       = np.array([p / 100 for p in PERCENTILES])   # ascending, for np.interp
IQB_COLS    = [f"iqb_p{p}" for p in PERCENTILES]
PCOLS       = IQB_COLS   # alias used in CI and plot sections
PCTS        = PROBS      # alias

# Raw metric column lists (retained from parquet for downstream analysis)
RAW_DL_COLS   = [f"download_p{p}" for p in PERCENTILES]
RAW_UL_COLS   = [f"upload_p{p}" for p in PERCENTILES]
RAW_LAT_COLS  = [f"latency_p{p}" for p in PERCENTILES]
RAW_LOSS_COLS = [f"loss_p{p}" for p in PERCENTILES]
RAW_METRIC_COLS = RAW_DL_COLS + RAW_UL_COLS + RAW_LAT_COLS + RAW_LOSS_COLS

METRICS = ["download", "upload", "latency", "loss"]

# IQB threshold bands per metric (from IQB config).
# "loose" = most permissive use case; "tight" = most demanding use case.
# These are the values a metric must surpass (download/upload: higher is better;
# latency/loss: lower is better) to contribute a passing score for each tier.
METRIC_THRESHOLDS = {
    "download": {"loose": 10,    "tight": 25,    "unit": "Mbps",   "higher_better": True},
    "upload":   {"loose": 5,     "tight": 25,    "unit": "Mbps",   "higher_better": True},
    "latency":  {"loose": 100,   "tight": 10,    "unit": "ms",     "higher_better": False},
    "loss":     {"loose": 0.01,  "tight": 0.005, "unit": "frac.",  "higher_better": False},
}

print(f"Percentile levels: {PERCENTILES}")
print(f"Raw metric columns: {len(RAW_METRIC_COLS)} total ({len(PERCENTILES)} percentiles × 4 metrics)")

## 2. Data Loading Helpers

Each calendar month lives in its own parquet file under `data/v1/`.  `load_month` merges
the download and upload tables on `country_code`, takes the minimum of their sample counts
(conservative: a country is only as well-measured as its worse-sampled direction), then
computes IQB at all 9 percentile levels.

The `iqb_range` column (max − min across percentile levels) is a quick proxy for how
spread out a country's IQB distribution is within a single month.

In [ ]:
def discover_periods(cache_root: Path) -> list[tuple[str, str, Path, Path]]:
    periods = []
    if not cache_root.exists():
        return periods
    for period_dir in sorted(cache_root.iterdir()):
        if not period_dir.is_dir():
            continue
        for end_dir in sorted(period_dir.iterdir()):
            dl = end_dir / "downloads_by_country" / "data.parquet"
            ul = end_dir / "uploads_by_country" / "data.parquet"
            if dl.exists() and ul.exists():
                s = period_dir.name[:8]
                e = end_dir.name[:8]
                start_str = f"{s[:4]}-{s[4:6]}-{s[6:8]}"
                end_str   = f"{e[:4]}-{e[4:6]}-{e[6:8]}"
                periods.append((start_str, end_str, dl, ul))
    return periods


def _country_name(cc: str) -> str:
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


def compute_iqb(row: pd.Series, p: int) -> float:
    data = {
        "m-lab": {
            "download_throughput_mbps": row[f"download_p{p}"],
            "upload_throughput_mbps":   row[f"upload_p{p}"],
            "latency_ms":               row[f"latency_p{p}"],
            "packet_loss":              row[f"loss_p{p}"],
        }
    }
    return calculator.calculate_iqb_score(data=data)


def load_month(start: str, dl_path: Path, ul_path: Path) -> pd.DataFrame | None:
    if dl_path.stat().st_size < 1000 or ul_path.stat().st_size < 1000:
        return None
    dl = pq.read_table(dl_path).to_pandas()
    ul = pq.read_table(ul_path).to_pandas()
    m = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))
    m["sample_count"] = m[["sample_count_down", "sample_count_up"]].min(axis=1)
    for p in PERCENTILES:
        m[f"iqb_p{p}"] = m.apply(compute_iqb, p=p, axis=1)
    m["iqb_range"] = m[IQB_COLS].max(axis=1) - m[IQB_COLS].min(axis=1)
    m["month"] = start
    # Retain raw metric columns alongside IQB scores for downstream analysis
    return m[["country_code", "month", "sample_count"] + IQB_COLS + ["iqb_range"] + RAW_METRIC_COLS]

print("Helpers defined.")

## 3. Load All 24 Months

In [ ]:
t0 = time.time()
all_periods = discover_periods(DATA_DIR)
periods = [(s, e, dl, ul) for s, e, dl, ul in all_periods]
START_DATE = periods[0][0] if periods else None
END_DATE   = periods[-1][1] if periods else None

print(f"Total cached months: {len(all_periods)}")
print(f"START_DATE = {START_DATE}  END_DATE = {END_DATE}")

frames = []
for idx, (s, e, dl, ul) in enumerate(periods, 1):
    f = load_month(s, dl, ul)
    status = "ok" if f is not None else "skipped (empty)"
    print(f"  [{idx}/{len(periods)}] {s}  {status}", flush=True)
    if f is not None:
        frames.append(f)

df = pd.concat(frames, ignore_index=True)
df["country_name"] = df["country_code"].map(_country_name)
df["month"] = pd.to_datetime(df["month"])
df = df.sort_values(["country_code", "month"]).reset_index(drop=True)

print(f"\nLoaded {len(df):,} rows "
      f"({df['country_code'].nunique()} countries × {df['month'].nunique()} months) "
      f"in {time.time() - t0:.1f}s")

## 4. CDF Merging

### Why merge in CDF space?

Naïvely averaging the p50 from January and p50 from February gives the wrong answer:
the result does not correspond to the p50 of the combined dataset.  The correct approach
is to average the *cumulative distribution functions* at every possible x-value, then read
off the percentiles of the merged CDF:

$$F_{\text{merged}}(x) = w_A \cdot F_A(x) + w_B \cdot F_B(x)$$

where $w_A = n_A / (n_A + n_B)$ and $w_B = n_B / (n_A + n_B)$.

### Implementation

`make_cdf(q, p)` reconstructs a CDF from the 9 known (value, probability) knots using
linear interpolation, with flat extrapolation beyond the tails.  Ties (months where the
IQB is the same at multiple percentile levels — a common artefact of score saturation at
0 or 1) are collapsed to a point mass.

`merge_two(qA, qB, nA, nB)` applies the formula above on a grid consisting of the union
of both months' percentile values, inverts the merged CDF to recover pooled percentiles,
and clips the result to lie within `[min(qA, qB), max(qA, qB)]` as a bracket check.
Any value outside that bracket would indicate a numerical error.

In [ ]:
def make_cdf(q, p):
    """Turn 9 (value, prob) points into a function: IQB value -> cumulative fraction."""
    q = np.asarray(q, float)
    uq = np.unique(q)                               # collapse tied values (floored CDFs)
    up = np.array([p[q == v].max() for v in uq])    # a tie = a point mass; keep top prob
    if len(uq) == 1:                                # fully flat month (n=1): a step function
        v = uq[0]
        return lambda x: (np.asarray(x, float) >= v).astype(float)
    return lambda x: np.interp(np.asarray(x, float), uq, up)   # flat tails automatically


def merge_two(qA, qB, nA, nB):
    """
    Pool two monthly IQB CDFs into one using sample-size-weighted CDF averaging.

    Parameters
    ----------
    qA, qB : array-like of length 9
        IQB percentile values (at PROBS levels) for month A and month B.
    nA, nB : int
        Sample counts for each month.

    Returns
    -------
    merged : ndarray of length 9
        Pooled IQB percentile values at PROBS levels, clipped to [min(qA,qB), max(qA,qB)].
    """
    wA, wB = nA / (nA + nB), nB / (nA + nB)
    FA, FB = make_cdf(qA, PCTS), make_cdf(qB, PCTS)
    x  = np.unique(np.concatenate([qA, qB]))       # grid = union of percentile values
    FC = wA * FA(x) + wB * FB(x)                   # pool in CDF-space
    keep   = np.concatenate([[True], np.diff(FC) > 0])   # ensure monotone for inversion
    merged = np.interp(PCTS, FC[keep], x[keep])    # invert: read off pooled percentiles
    return np.clip(merged, np.minimum(qA, qB), np.maximum(qA, qB))

print("CDF merge functions defined.")

In [ ]:
def nonoverlapping_pools(sub: pd.DataFrame, k: int = 2,
                         require_adjacent: bool = True) -> pd.DataFrame:
    """
    Group a country's months into consecutive non-overlapping blocks of size k
    and pool each block via weighted CDF averaging.

    If require_adjacent=True, a block is only kept when its months are
    calendar-consecutive (no gaps). Skipped blocks are reported.
    """
    sub = sub.sort_values("month").reset_index(drop=True)
    rows, skipped = [], []
    for i in range(0, len(sub) - k + 1, k):          # drop trailing partial block
        block = sub.iloc[i:i + k]
        months = block["month"].dt.to_period("M").astype("int64")
        gaps = (months.diff().dropna() != 1).any()
        if require_adjacent and gaps:
            skipped.append(" + ".join(m.strftime("%Y-%m") for m in block["month"]))
            continue
        q_list = [block.iloc[j][IQB_COLS].to_numpy(float) for j in range(k)]
        n_list = block["sample_count"].to_numpy(float)
        pooled = pool_months(q_list, n_list)
        rows.append({
            "start": block["month"].iloc[0],
            "end": block["month"].iloc[-1],
            "n_total": n_list.sum(),
            **{c: v for c, v in zip(IQB_COLS, pooled)},
        })
    if skipped:
        print(f"  skipped {len(skipped)} non-adjacent block(s): {skipped}")
    return pd.DataFrame(rows)


def plot_month_vs_pooled(cc: str, k: int = 2):
    sub = df[df["country_code"] == cc]
    pooled = nonoverlapping_pools(sub, k)
    fig, ax = plt.subplots(figsize=(8, 5))
    for _, r in sub.iterrows():                       # per-month CDFs, faint
        ax.plot(r[IQB_COLS].to_numpy(float), PCTS, color="grey", alpha=0.3, lw=1)
    for _, r in pooled.iterrows():                    # pooled CDFs, bold
        ax.plot(r[IQB_COLS].to_numpy(float), PCTS,
                lw=2, label=f"{r['start']:%Y-%m}+{r['end']:%Y-%m}")
    ax.set_xlabel("IQB"); ax.set_ylabel("cumulative fraction")
    ax.set_title(f"{_country_name(cc)}: monthly (grey) vs {k}-month pooled CDFs")
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout(); plt.show()

plot_month_vs_pooled("US")
plot_month_vs_pooled("TV")

## 5. Building Two-Month Blocks

Within each country's time series, months are assigned a sequential index (`seq`) and
then grouped into non-overlapping pairs: months 0–1 → block 0, months 2–3 → block 1,
and so on.  Countries with an odd number of months get one single-month block at the end.

With 24 months, every country should produce 12 clean pairs and 0 leftovers — except
countries that were absent from some months (not all 245 countries appear in every month).

In [ ]:
df["seq"]   = df.groupby("country_code").cumcount()   # 0-based rank within country
df["block"] = df["seq"] // 2                           # consecutive pairs

# Sanity check: how many countries have an odd number of months?
month_counts = df.groupby("country_code")["month"].nunique()
odd_countries = month_counts[month_counts % 2 == 1]
print(f"Countries with an odd number of months (will have one single-month block): "
      f"{len(odd_countries)}")
print(f"Example: {odd_countries.head().to_dict()}")

## 6. Pooling Each Block

For each `(country_code, block)` group:
- If the group has two months: call `merge_two` using both months' IQB vectors and sample counts.
- If only one month: carry it forward unchanged (flagged as `single_month=True`).

The result is `pooled`: one row per (country, block) with the merged 9-percentile IQB vector
and the summed sample count.

In [ ]:
rows = []
for (cc, b), g in df.groupby(["country_code", "block"]):
    g = g.sort_values("month")
    if len(g) == 1:
        q      = g[PCOLS].to_numpy()[0]
        n      = int(g["sample_count"].iloc[0])
        single = True
    else:
        qA, qB = g[PCOLS].to_numpy()
        nA, nB = g["sample_count"].to_numpy()
        q      = merge_two(qA, qB, nA, nB)
        n      = int(nA + nB)
        single = False
    rows.append({
        "country_code": cc,
        "block":        b,
        "block_start":  g["month"].iloc[0],
        "sample_count": n,
        "single_month": single,
        **dict(zip(PCOLS, q)),
    })

pooled = pd.DataFrame(rows)

n_single = pooled["single_month"].sum()
print(f"Total blocks: {len(pooled):,}")
print(f"Single-month blocks (odd leftovers): {n_single}")
print()
# Countries that contributed a single-month block
print("Countries with a single-month block:")
print(pooled[pooled["single_month"]][["country_code", "block", "sample_count"]])

## 7. Metric Contribution Analysis

### Motivation

`iqb_range` (max − min IQB across all 9 percentile levels) tells us *that* a country's
score varies, but not *why*.  Because IQB is a weighted pass/fail against fixed thresholds,
variation across percentile levels comes from one or more raw metrics crossing a threshold
between, say, p25 and p75.  That crossing is "natural" when the underlying metric genuinely
degrades as more users saturate the network — and is distinguishable from statistical noise
by inspecting the raw values.

### Method: counterfactual marginal contribution

For each metric `m` at each percentile level `p`, we compute:

$$\text{contrib}_{m,p} = \text{IQB}_{\text{best}_m}(p) - \text{IQB}_{\text{actual}}(p)$$

where $\text{IQB}_{\text{best}_m}$ is the score obtained when metric `m` is replaced by a
perfect value (download/upload → 1000 Mbps; latency → 0.1 ms; loss → 0) while all other
metrics remain at their actual values.  A large `contrib` means that metric is the binding
constraint at that percentile.  By construction `contrib ≥ 0` and the sum across all four
metrics is an upper bound (not an exact decomposition) because the pass/fail rule is
non-linear.

The **bottleneck** metric at each (country, month, percentile) is the one with the largest
marginal contribution.

In [ ]:
_BEST = {"download": 1000.0, "upload": 1000.0, "latency": 0.1, "loss": 0.0}
_RAW_KEY = {
    "download": "download_throughput_mbps",
    "upload":   "upload_throughput_mbps",
    "latency":  "latency_ms",
    "loss":     "packet_loss",
}

def _iqb_best_m(row: pd.Series, p: int, exclude: str) -> float:
    """IQB at percentile p with metric `exclude` replaced by its best-case value."""
    data = {"m-lab": {
        _RAW_KEY[m]: (_BEST[m] if m == exclude else row[f"{m}_p{p}"])
        for m in METRICS
    }}
    # latency and loss columns in parquet are named "latency_p{p}" and "loss_p{p}"
    # but the upload column is "upload_p{p}" — all four are in df after load_month
    return calculator.calculate_iqb_score(data=data)


print("Computing marginal metric contributions across all rows and percentile levels …")
t0 = time.time()

for metric in METRICS:
    for p in PERCENTILES:
        df[f"contrib_{metric}_p{p}"] = df.apply(_iqb_best_m, p=p, exclude=metric, axis=1) - df[f"iqb_p{p}"]

print(f"Done in {time.time() - t0:.1f}s")
print(f"New columns added: {4 * len(PERCENTILES)}  (e.g. contrib_download_p50, contrib_latency_p25, …)")
print(f"df shape: {df.shape}")

# Quick sanity: contributions must be ≥ 0 everywhere
neg_check = sum((df[f"contrib_{m}_p{p}"] < -1e-9).sum()
                for m in METRICS for p in PERCENTILES)
print(f"Rows with negative contribution (should be 0): {neg_check}")

In [ ]:
# Identify the bottleneck metric (largest marginal contribution) at each percentile level.
# When all contributions are 0 the country has a perfect IQB at that level; label it "none".
for p in PERCENTILES:
    contrib_cols = [f"contrib_{m}_p{p}" for m in METRICS]
    max_contrib  = df[contrib_cols].max(axis=1)
    bottleneck   = df[contrib_cols].idxmax(axis=1).str.replace(rf"contrib_|_p{p}", "", regex=True)
    df[f"bottleneck_p{p}"] = bottleneck.where(max_contrib > 1e-6, "none")

print("Bottleneck metric per percentile level added.")
print("\nBottleneck distribution at p50 (globally, all country-months):")
print(df["bottleneck_p50"].value_counts().to_string())

print("\nBottleneck distribution at p25:")
print(df["bottleneck_p25"].value_counts().to_string())

print("\nBottleneck distribution at p75:")
print(df["bottleneck_p75"].value_counts().to_string())

## 8. Case Study: Raw Metrics + IQB Across Percentile Levels

Each panel shows a single metric's value at the 9 percentile levels alongside the loose
and tight IQB thresholds for that metric (shaded bands).  The top panel shows IQB itself.

Reading the chart tells you *why* IQB is high at p1 and lower at p99 (or vice-versa):
the metric whose value crosses a threshold band is the one causing the IQB drop.  Natural
variation produces a smooth, monotone metric profile; measurement noise produces a jagged
one that does not respect the threshold bands consistently.

In [ ]:
_METRIC_LABELS = {
    "download": "Download (Mbps)",
    "upload":   "Upload (Mbps)",
    "latency":  "Latency (ms)",
    "loss":     "Packet loss (fraction)",
}
_METRIC_COLORS = {
    "download": "steelblue",
    "upload":   "seagreen",
    "latency":  "darkorange",
    "loss":     "crimson",
}


def plot_raw_metrics_vs_iqb(cc: str, months: list | None = None, avg_months: bool = True):
    """
    5-panel figure for country `cc`:
      - Top panel: IQB across the 9 percentile levels (one line per month, or averaged).
      - Panels 2-5: raw metric value across the 9 percentile levels with threshold bands.

    Parameters
    ----------
    months : list of str (YYYY-MM) or None — filter to specific months; None = all.
    avg_months : if True, overlay the per-month lines in grey and the mean in colour.
    """
    sub = df[df["country_code"] == cc].copy()
    if months:
        sub = sub[sub["month"].dt.strftime("%Y-%m").isin(months)]
    if sub.empty:
        print(f"No data for {cc}")
        return

    x = PERCENTILES   # x-axis: percentile level (1, 5, 10, …, 99)

    fig, axes = plt.subplots(5, 1, figsize=(9, 16), sharex=True)

    # --- Panel 0: IQB ---
    ax = axes[0]
    for _, row in sub.iterrows():
        iqb_vals = row[IQB_COLS].to_numpy(float)
        ax.plot(x, iqb_vals, color="grey", alpha=0.25, lw=1)
    mean_iqb = sub[IQB_COLS].mean().to_numpy(float)
    ax.plot(x, mean_iqb, color="black", lw=2.5, marker="o", label="mean IQB")
    ax.set_ylabel("IQB score")
    ax.set_ylim(-0.02, 1.05)
    ax.set_title(f"IQB across percentile levels", fontsize=10)
    ax.axhline(0, color="black", lw=0.5, ls="--")
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.35)

    # --- Panels 1-4: raw metrics ---
    for ax, metric in zip(axes[1:], METRICS):
        raw_cols = [f"{metric}_p{p}" for p in PERCENTILES]
        thr      = METRIC_THRESHOLDS[metric]
        color    = _METRIC_COLORS[metric]
        hb       = thr["higher_better"]

        # Per-month lines in grey
        for _, row in sub.iterrows():
            vals = row[raw_cols].to_numpy(float)
            ax.plot(x, vals, color="grey", alpha=0.25, lw=1)

        # Mean across months
        mean_vals = sub[raw_cols].mean().to_numpy(float)
        ax.plot(x, mean_vals, color=color, lw=2.5, marker="o",
                label=f"mean {_METRIC_LABELS[metric]}")

        # Threshold bands
        lo, hi = thr["tight"], thr["loose"]
        if not hb:            # latency / loss: lower is better, thresholds are upper limits
            lo, hi = thr["loose"], thr["tight"]   # loose=100ms, tight=10ms → band 10..100

        # Colour the band between the two thresholds
        ax.axhspan(lo, hi, alpha=0.12, color=color, label=f"threshold band ({lo}–{hi} {thr['unit']})")
        ax.axhline(lo, color=color, lw=1.0, ls="--", alpha=0.7)
        ax.axhline(hi, color=color, lw=1.0, ls=":",  alpha=0.7)

        ax.set_ylabel(_METRIC_LABELS[metric])
        ax.set_title(f"{_METRIC_LABELS[metric]} — threshold band shaded", fontsize=10)
        ax.legend(fontsize=8)
        ax.grid(True, linestyle="--", alpha=0.35)

    axes[-1].set_xlabel("Percentile level")
    fig.suptitle(
        f"{_country_name(cc)} ({cc}) — Raw metrics and IQB across percentile levels\n"
        f"grey = individual months, coloured = mean  |  n months = {len(sub)}",
        fontsize=12, y=1.005,
    )
    fig.tight_layout()
    plt.show()


# ── Case studies ──────────────────────────────────────────────────────────────
plot_raw_metrics_vs_iqb("US")     # well-sampled; should show clean monotone profiles
plot_raw_metrics_vs_iqb("TV")     # sparse; jagged profiles expected
plot_raw_metrics_vs_iqb("ER")     # Eritrea: appeared as wide-CI country at p75

## 9. Global Bottleneck Analysis

### 9a. What fraction of country-months are bottlenecked by each metric?

At each percentile level, we ask: across all country-months, how often is each metric
the binding constraint on IQB?  This reveals whether IQB variation is primarily driven
by download throughput (a capacity problem), latency (a routing/congestion problem),
packet loss (a reliability problem), or upload (often overlooked).

### 9b. Does IQB range correlate with metric bottleneck strength?

`iqb_range` (the spread of IQB scores across the 9 percentile levels within a single
country-month) quantifies *how much* IQB varies.  We check whether countries with high
`iqb_range` are those where at least one metric straddles a threshold — confirming that
IQB instability across percentile levels is driven by threshold crossings, not noise.

In [ ]:
# ── 9a: Bottleneck frequency heatmap ─────────────────────────────────────────
# For each percentile level, compute fraction of country-months bottlenecked by each metric.
bottleneck_cols = [f"bottleneck_p{p}" for p in PERCENTILES]
all_labels      = METRICS + ["none"]
color_map       = {**_METRIC_COLORS, "none": "lightgrey"}

freq = {}
for p in PERCENTILES:
    col   = f"bottleneck_p{p}"
    vc    = df[col].value_counts(normalize=True)
    freq[p] = {lbl: vc.get(lbl, 0.0) for lbl in all_labels}

freq_df = pd.DataFrame(freq, index=all_labels).T   # shape: (9 percentile levels) × (5 labels)

fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(PERCENTILES))
x_pos  = np.arange(len(PERCENTILES))
for lbl in all_labels:
    vals = freq_df[lbl].to_numpy()
    ax.bar(x_pos, vals, bottom=bottom, label=lbl, color=color_map[lbl], alpha=0.85)
    bottom += vals

ax.set_xticks(x_pos)
ax.set_xticklabels([f"p{p}" for p in PERCENTILES])
ax.set_ylabel("Fraction of country-months")
ax.set_xlabel("Percentile level")
ax.set_title("Bottleneck metric distribution across percentile levels\n(stacked bars, all country-months)")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, 1)
ax.grid(axis="y", linestyle="--", alpha=0.35)
plt.tight_layout()
plt.show()

print("\nFraction of country-months bottlenecked by each metric (rows = metric, cols = percentile):")
print(freq_df[METRICS].round(3).to_string())

In [ ]:
# ── 9b: IQB range vs. max metric contribution ────────────────────────────────
# For each country-month, "max_contrib" = the single largest marginal contribution
# across all metrics and all percentile levels.  This quantifies how much a single
# metric is constraining IQB somewhere in the percentile distribution.
all_contrib_cols = [f"contrib_{m}_p{p}" for m in METRICS for p in PERCENTILES]
df["max_contrib"] = df[all_contrib_cols].max(axis=1)

# Scatter: iqb_range vs max_contrib, coloured by log10(sample_count)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter of all country-months
ax = axes[0]
sc = ax.scatter(df["max_contrib"], df["iqb_range"],
                c=np.log10(df["sample_count"].clip(1)),
                cmap="viridis", alpha=0.35, s=12, rasterized=True)
plt.colorbar(sc, ax=ax, label="log₁₀(sample count)")
ax.set_xlabel("Max metric contribution (any metric, any percentile)")
ax.set_ylabel("IQB range (max IQB − min IQB across percentile levels)")
ax.set_title("IQB range vs. max metric bottleneck strength\nColour = log₁₀(n)")
ax.grid(True, linestyle="--", alpha=0.35)

# Overlay correlation
r = df[["max_contrib", "iqb_range"]].dropna().corr().iloc[0, 1]
ax.text(0.05, 0.93, f"r = {r:.3f}", transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle="round", fc="white", alpha=0.8))

# Right: mean IQB range per bottleneck metric (at p50) — is the bottleneck metric
# predictive of how unstable IQB is across percentile levels?
ax2 = axes[1]
means = (df.groupby("bottleneck_p50")["iqb_range"]
           .agg(["mean", "median", "count"])
           .reindex(all_labels)
           .dropna())
bars = ax2.bar(means.index, means["mean"],
               color=[color_map.get(lbl, "grey") for lbl in means.index],
               alpha=0.85, label="mean IQB range")
ax2.errorbar(means.index, means["mean"],
             yerr=means["mean"] * 0.0,   # placeholder; could add std
             fmt="none", color="black", capsize=3)
for i, (lbl, row) in enumerate(means.iterrows()):
    ax2.text(i, row["mean"] + 0.005, f'n={int(row["count"])}', ha="center", fontsize=7)
ax2.set_xlabel("Bottleneck metric at p50")
ax2.set_ylabel("Mean IQB range")
ax2.set_title("Mean IQB range by bottleneck metric (at p50)")
ax2.grid(axis="y", linestyle="--", alpha=0.35)

plt.tight_layout()
plt.show()

print("Correlation (max_contrib, iqb_range):", round(r, 4))
print("\nMean IQB range by p50 bottleneck metric:")
print(means[["mean", "median", "count"]].round(4).to_string())

## 7. Confidence Intervals

### Formula

The standard error of the $p$-th sample quantile from $n$ indepwendent observations is:

$$\text{SE}(\hat{Q}_p) \approx \frac{\sqrt{p(1-p)/n}}{f(Q_p)}$$

where $f(Q_p)$ is the density of the distribution at the $p$-th quantile.  Because we
only have the CDF at 9 knot points — not a continuous density — we approximate this by
shifting the *probability level* rather than the value:

1. Compute the rank-noise half-width: $\delta_p = z \cdot \sqrt{p(1-p)/n}$
2. Shift the probability up and down: $p^\pm = \text{clip}(p \pm \delta_p, 0.01, 0.99)$
3. Read the IQB value at those shifted probabilities from the CDF: $\hat{Q}_{p^\pm}$

This is equivalent to asking: "if the true rank were $p \pm \delta_p$ instead of $p$,
what IQB value would that correspond to?"  The resulting band is wider where the CDF
is steep (uncertain region) and narrower where it is flat (score is pinned at 0 or 1).

We compute CIs for both the original monthly data (`df_ci`) and the pooled blocks
(`pooled_ci`).

In [ ]:
def quantile_ci(row, z=1.96):
    """Compute 95% CI bounds for each of the 9 IQB percentile estimates."""
    n   = row["sample_count"]
    iqb = row[PCOLS].to_numpy(dtype=float)

    se_p  = np.sqrt(PCTS * (1 - PCTS) / n)          # rank noise half-width (fraction)
    p_lo  = np.clip(PCTS - z * se_p, 0.01, 0.99)    # shifted probability levels
    p_hi  = np.clip(PCTS + z * se_p, 0.01, 0.99)

    iqb_lo = np.interp(p_lo, PCTS, iqb)             # read IQB at those positions
    iqb_hi = np.interp(p_hi, PCTS, iqb)

    return pd.Series({
        **{f"{c}_lo": lo for c, lo in zip(PCOLS, iqb_lo)},
        **{f"{c}_hi": hi for c, hi in zip(PCOLS, iqb_hi)},
    })


ci_monthly = df.apply(quantile_ci, axis=1)
df_ci      = pd.concat([df, ci_monthly], axis=1)

ci_pooled  = pooled.apply(quantile_ci, axis=1)
pooled_ci  = pd.concat([pooled, ci_pooled], axis=1)

# Derive CI width for each percentile level
for c in PCOLS:
    pooled_ci[f"{c}_width"] = pooled_ci[f"{c}_hi"] - pooled_ci[f"{c}_lo"]
    df_ci[f"{c}_width"]     = df_ci[f"{c}_hi"]     - df_ci[f"{c}_lo"]

print(f"df_ci shape:     {df_ci.shape}")
print(f"pooled_ci shape: {pooled_ci.shape}")

## 8. Which Blocks Have the Widest Confidence Intervals?

CI width at p50 is a direct proxy for score reliability: a width of 0.9 means the 95%
interval spans almost the entire IQB scale, making the point estimate nearly meaningless.
We expect the widest bands to come from countries with very small sample counts.

In [ ]:
for pct in [25, 50, 75]:
    col = f"iqb_p{pct}"
    width_col = f"{col}_width"
    print(f"\nBlocks with widest {col} confidence intervals (most uncertain):")
    print(
        pooled_ci.sort_values(width_col, ascending=False)[
            ["country_code", "block", "sample_count",
             f"{col}_lo", col, f"{col}_hi", width_col]
        ].head(10).to_string(index=False)
    )


## 9. Case Study: Tuvalu (TV)

Tuvalu is one of the sparsest countries in the dataset — some months have as few as 1
measurement.  It is a useful stress-test: does pooling two months together actually
improve the CI width compared to looking at each month individually?

The answer here is **mixed**.  Pooling does increase the sample count (and should narrow
the CI in theory), but if one of the two months has a very different score from the other,
the merged distribution is wider and the CI may not shrink — or may even grow — because
the merged CDF spans a larger IQB range.

This illustrates why the stability gate in notebook 07 matters: pooling is only beneficial
when the underlying distribution is stable across the window.

In [ ]:
SPARSE_CC = "TV"   # Tuvalu
print(f"Country: {_country_name(SPARSE_CC)} ({SPARSE_CC})\n")

sub_before = df_ci[df_ci["country_code"] == SPARSE_CC]
sub_after  = pooled_ci[pooled_ci["country_code"] == SPARSE_CC]

for pct in [25, 50, 75]:
    col = f"iqb_p{pct}"
    print(f"========================= p{pct} =========================")

    before = sub_before[
        ["month", "sample_count", f"{col}_lo", col, f"{col}_hi", f"{col}_width"]
    ].copy()

    after = sub_after[
        ["block", "block_start", "sample_count", "single_month",
         f"{col}_lo", col, f"{col}_hi", f"{col}_width"]
    ].copy()

    print("BEFORE (monthly, non-pooled):")
    print(before.to_string(index=False))
    print()
    print("AFTER (2-month blocks, CDF-pooled):")
    print(after.to_string(index=False))
    print()
    print(f"Mean p{pct} CI width — monthly: {before[f'{col}_width'].mean():.3f} "
          f" | pooled: {after[f'{col}_width'].mean():.3f}")
    print()

## 10. Visualisation: Full IQB CDF with Confidence Bands

Each panel shows the IQB CDF (x-axis = percentile level, y-axis = IQB score) for one
2-month block, along with the 95% CI band.  A narrow band = reliable estimate.  A wide
band = the score could plausibly be almost anywhere on the 0–1 scale.

Note how blocks 4 and 5 (mid-2025, n ≈ 29 and 13) are noticeably tighter than blocks
0–2 (early 2024, n ≈ 6–19).

In [ ]:
cc  = SPARSE_CC
sub = pooled_ci[pooled_ci["country_code"] == cc].sort_values("block")

x   = PCTS * 100                           # 1, 5, 10 … 99
los = [f"{c}_lo" for c in PCOLS]
his = [f"{c}_hi" for c in PCOLS]

n_blocks = len(sub)
fig, axes = plt.subplots(n_blocks, 1, figsize=(4, 3 * n_blocks), sharey=True)
if n_blocks == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, sub.iterrows()):
    mid = row[PCOLS].to_numpy(float)
    lo  = row[los].to_numpy(float)
    hi  = row[his].to_numpy(float)
    ax.fill_between(x, lo, hi, alpha=0.25, color="steelblue", label="95% CI")
    ax.plot(x, mid, marker="o", lw=1.5, color="steelblue", label="IQB")
    ax.set_title(
        f"Block {int(row['block'])}  "
        f"({row['block_start'].strftime('%b %Y')}–next)  "
        f"n={row['sample_count']}",
        fontsize=9,
    )
    ax.set_xlabel("Percentile")
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle="--", alpha=0.35)

axes[0].set_ylabel("IQB score")
axes[0].legend(fontsize=8)
fig.suptitle(
    f"{_country_name(cc)} ({cc}) — Pooled 2-month blocks\n"
    "IQB CDF with 95% CI bands",
    fontsize=11,
)
fig.tight_layout()
plt.show()

### 10b. Monthly (Non-Pooled) View for Comparison

The same country plotted month-by-month, without any pooling.  Months with n=1 show a
completely flat CI (no uncertainty can be estimated from a single observation) — an
artefact of the formula, not a sign of reliability.

In [ ]:
sub_m = df_ci[df_ci["country_code"] == cc].sort_values("month")

n_panels = len(sub_m)
fig, axes = plt.subplots(n_panels, 1, figsize=(4, 2.6 * n_panels), sharey=True)
if n_panels == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, sub_m.iterrows()):
    mid = row[PCOLS].to_numpy(float)
    lo  = row[los].to_numpy(float)
    hi  = row[his].to_numpy(float)
    ax.fill_between(x, lo, hi, alpha=0.25, color="tomato")
    ax.plot(x, mid, marker="o", lw=1.5, color="tomato")
    ax.set_title(
        f"{row['month'].strftime('%Y-%m')}  n={row['sample_count']}",
        fontsize=8,
    )
    ax.set_xlabel("Percentile")
    ax.set_ylim(0, 1)
    ax.grid(True, linestyle="--", alpha=0.35)

axes[0].set_ylabel("IQB score")
fig.suptitle(
    f"{_country_name(cc)} ({cc}) — Monthly (non-pooled)\n"
    "IQB CDF with 95% CI bands",
    fontsize=11,
)
fig.tight_layout()
plt.show()